In [ ]:
!git clone https://github.com/CfM47/ML-Project.git

In [ ]:
import sys
from pathlib import Path

In [ ]:
!ls /kaggle/input/segmentations-images-automl/pictures

In [ ]:
segmentation_root = Path('/kaggle/input/segmentations-images-automl/pictures')

TRAIN_LABELED = segmentation_root / 'vega_3_tescan_labeled_images'
TRAIN_UNLABELED = segmentation_root / 'vega_3_tescan_unlabeled_images'
TEST_UNLABELED = segmentation_root / 'sampled_unlabeled'
TEST_LABELED = segmentation_root / 'sampled_labeled'

WORKING_DIR = Path('/kaggle/working')
PROJECT_ROOT = WORKING_DIR / 'ML-Project'
OUTPUT_DIR = WORKING_DIR / 'swin_results'

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from model.swin.config import SwinTrainingConfig

config = SwinTrainingConfig(
    output_dir=OUTPUT_DIR,
    device='auto',
)

In [ ]:
from model.swin.train import run_final_training

result = run_final_training(
    TRAIN_UNLABELED,
    TRAIN_LABELED,
    TEST_UNLABELED,
    TEST_LABELED,
    config=config,
)

In [ ]:
result.predictions_figure

In [ ]:
if result.loss_curves_figure is not None:
    display(result.loss_curves_figure)
else:
    print('Loss curves not available (training history validation failed)')

In [ ]:
print("Full Test Set Metrics:")
for metric_name, value in result.test_metrics.full_metrics.items():
    print(f"  {metric_name}: {value:.4f}")

print("\nLeave-Two-Out Summary:")
print(f"  Subsets: {result.test_metrics.num_subsets} of size {result.test_metrics.subset_size}")
for metric_name in sorted(result.test_metrics.subset_means.keys()):
    mean_val = result.test_metrics.subset_means[metric_name]
    std_val = result.test_metrics.subset_stds[metric_name]
    print(f"  {metric_name}: {mean_val:.4f} ± {std_val:.4f}")

In [ ]:
# Display all histograms
for metric_name, fig in sorted(result.histograms.items()):
    print(f'\n{metric_name}:')
    display(fig)